![iSA logo](../iSA_logo.png)
### Materiały do zajęć Statystyka
#### Kurs Junior Data Scientist

**Wojciech Artichowicz**

Wykonano $193$ pomiary cechy pewnego zjawiska. Dane w postaci szeregu przedziałowego znajdują się w tabeli. Przy pomocy testu $\chi^2$ sprawdzić czy próba pochodzi z populacji o rozkładzie normalnym.

In [1]:
import numpy as np
import scipy as sp
import scipy.stats as st
import matplotlib.pyplot as plt
%matplotlib inline

**Wprowadzenie danych**

In [2]:
gr_klas = np.arange(0,1.61,0.2) # generowanie granic klas
print(gr_klas)
ni = np.array([32, 55, 45, 36, 12, 8, 3, 2]) # częstości klas określone w zadaniu
print(ni)
N = sum(ni) # suma wszystkich elementów
print(N)

[0.  0.2 0.4 0.6 0.8 1.  1.2 1.4 1.6]
[32 55 45 36 12  8  3  2]
193


Test $\chi^2$ można przeprowadzać tylko, jeśli żadna z klas nie liczy mniej niż 5 elementów. Jeśli jest inaczej to należy scalić sąsiadujące klasy.

In [3]:
ni<5

array([False, False, False, False, False, False,  True,  True])

**Utworzenie zmodyfikowanego wektora liczności oraz granic klas.**

Jako, że liczność w niektórych klasach jest mniejsza niż 5, konieczne jest scalenie sąsiadujących klas.

In [4]:
ni_mod = np.delete(ni,-1) # utworzenie zmodyfikowanego wektora liczności - krótszego o jeden element od ni
ni_mod[-1] += ni[-1] # zsumowanie liczebności dwóch ostatnich klas
gr_klas_mod = np.delete(gr_klas,-2) # usunięcie przedostatniego elementu wektora granic klas, 
                                    # tj. scalenie dwóch ostatnich klas
print(ni_mod)
print(gr_klas_mod)

[32 55 45 36 12  8  5]
[0.  0.2 0.4 0.6 0.8 1.  1.2 1.6]


**Jako, że sprawdzeniu ma podlegać hipoteza, iż próba pochodzi z rozkładu normalnego, konieczne jest estymownanie parametrów tego rozkładu na podstawie próby.**

W związku z tym, że posiadane dane mają postać szeregu rozdzielczego, konieczne jest utworzenie wektora zawierającego wartości środków klas i obliczenie średniej i odchylenia standardowego przy użyciu wzorów dla szeregu rozdzielczego.

In [5]:
xs = np.cumsum(np.diff(gr_klas_mod))-np.diff(gr_klas_mod)/2 # środki klas histogramu

m = sum(xs*ni_mod)/N # obliczenie średniej (wzór dla szeregu rozdzielczego)
s = np.sqrt(sum(ni_mod*(xs-m)**2)/(N-1)) # obliczenie odchylenia standardowego (wzór dla seregu rozdzielczego, estymator obc.)

Jeżeli ogony rozkładu teoretycznego (tu: rozkładu normalnego) osiągają wartości w nieskończoności to pierwszą i/lub ostatnią klasę nalezy otworzyć i zsumowac całe pole w kierunku plus lub minus nieskończoności.

W tym celu wygodnie jest jako granicę pierwszej klasy określić $-\infty$, a ostatniej $+\infty$

In [6]:
gr_klas_mod[0]=-np.inf
gr_klas_mod[-1]=+np.inf

Następnym krokiem jest obliczenie wartości prawdopodobieństw osiągnięcia wartości w danej klasie przez zmienną losową o rozkładzie teoretycznym. Posiadając zmodyfikowany wektor granic klas wystarczy obliczyć wartości dystrybuanty i odjąć je parami (funkcja numpy.diff). 

In [7]:
rozkladNormalny = st.norm(m,s)
F = rozkladNormalny.cdf(gr_klas_mod)
p = np.diff(F)

Mnożąc otrzymane prawdopodobieństwa przez liczebność próby, otrzymuje się teoretyczne liczebności w klasach, które powinny wystąpić, gdyby próba pochodziła z rozważanego rozkładu.

In [8]:
nt = p*N

Kolejnym krokiem jest obliczenie statystki testowej: $$\chi_t^2=\sum_{j=1}^{j=k} \frac{(n_j-n_{t,j})^2}{n_{t,j}}$$
gdzie $n_j$ jest częstością obserwowaną w $j$-tej klasie, $n_{t,j}$ jest częstością teoretyczną, $k$ oznacza liczbę klas (z uwzględnieniem połączenia klas), natomiast $\chi_t^2$ jest statystyką testową.

In [9]:
chi2t = sum((ni_mod-nt)**2/nt)
chi2t

14.570384971180175

Ostatnim krokiem jest policzenie p-wartości. Jako, że w testach tego typu (tzn. badania odchyłki, błędu, odstawania, itp.) minimalną wartością jest $0$, a im większa różnica między obserwacjami a oczekiwaniami tym statystyka rośnie, bada się tylko prawostronną hipotezę alternatywną. W związku z tym dla testu zgodności $\chi^2$ obszar krytyczny jest prawostronny, a p-wartość zawsze liczy się od prawej strony, tzn. $$p_v=1-F(\chi_t^2)$$

In [10]:
k = len(ni_mod) # liczba klas
m = 2 # liczba parametrów rozkładu teoretycznego obliczonych na podstawie próby, tu: średnia i odchylenie std.
rozkladChi2 = st.chi2(k-m-1) # utworzenie instancji obiektu reprezentującego rozkład chi2
pvalue = 1-rozkladChi2.cdf(chi2t) # obliczenie p-wartości
pvalue

0.005680462622405735

Uzyskana p-wartość jest mała $(p_v<\alpha)$ w porównaniu do poziomu istotności, zatem hipotezę zerową należy odrzucić. Oznacza to, że badana próba nie pochodzi z populacji o rozkładzie normalnym.

Biblioteka scipy.stats oferuje test $\chi^2$ w postaci funkcji [`st.chisquare()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chisquare.html#scipy.stats.chisquare), któa pobiera argumenty w postaci częstości obserwowanch, częstości teoretycznych oraz delty liczby stopni swobody. Liczba stopni swobody obliczana jest jako $df=k-\Delta-1$. W analizowanym przypadku wartość $\Delta$ równa jest $m$.

In [11]:
st.chisquare(ni_mod,nt,ddof=m)

Power_divergenceResult(statistic=14.570384971180175, pvalue=0.00568046262240577)